# 🤖 ZackAI — Kaggle Training Notebook

Train a custom general-assistant AI model called **ZackAI** using [Unsloth](https://github.com/unslothai/unsloth).

- **Base model**: Qwen 2.5 3B
- **Dataset**: teknium/OpenHermes-2.5
- **Output**: ~4 GB GGUF (Q4_K_M) — ready for LM Studio
- **Auto-upload**: model is saved to your Hugging Face account so nothing is lost when the Kaggle session ends

**Estimated runtime on Kaggle free T4 GPU**: ~4–6 hours

## 🔑 Step 1 — Configuration

**Edit the two variables below before running any other cell.**

In [ ]:
# ============================================================
#  🔑 CONFIGURATION — Set Your Credentials Here
# ============================================================

HF_TOKEN    = "hf_YOUR_TOKEN_HERE"        # Hugging Face write token
HF_USERNAME = "your-huggingface-username"  # Your HF username

# ============================================================
#  (Optional) Training tweaks
# ============================================================
MAX_STEPS   = 1000   # Increase for better quality (costs more time)
MAX_SEQ_LEN = 2048   # Context window

HF_REPO = f"{HF_USERNAME}/ZackAI"
print(f"Model will be uploaded to: https://huggingface.co/{HF_REPO}")

## 📦 Step 2 — Install Dependencies

In [ ]:
%%capture
# Install Unsloth and all required libraries
!pip install unsloth
!pip install --upgrade trl transformers accelerate datasets bitsandbytes huggingface_hub peft

## 📚 Step 3 — Imports

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import login
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 🔐 Step 4 — Log in to Hugging Face

In [ ]:
login(token=HF_TOKEN)
print("✅ Logged in to Hugging Face")

## 🧠 Step 5 — Load Base Model (Qwen 2.5 3B, 4-bit)

Qwen 2.5 3B quantizes to ~4 GB at Q4_K_M, which is ideal for LM Studio.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # Auto-detect (fp16 on T4)
    load_in_4bit=True,   # 4-bit quantization for memory efficiency
)
print("✅ Base model loaded")

## 🔧 Step 6 — Apply LoRA Adapters

LoRA lets us train a small set of adapter weights instead of the full model, making it feasible on a free T4 GPU.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Saves VRAM
    random_state=3407,
)
print("✅ LoRA adapters applied")

## 📊 Step 7 — Load & Format the Dataset

We use [teknium/OpenHermes-2.5](https://huggingface.co/datasets/teknium/OpenHermes-2.5), a high-quality general-assistant dataset.

Each example is formatted using the **ChatML** template that Qwen expects, with the system prompt setting the ZackAI identity.

In [ ]:
SYSTEM_PROMPT = (
    "You are ZackAI, a helpful, harmless, and honest AI assistant created by Zack."
)

def format_conversation(example):
    """Convert an OpenHermes conversation into the ChatML format Qwen uses."""
    conversations = example.get("conversations", [])

    text = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"

    for turn in conversations:
        role = turn.get("from", "").strip().lower()
        content = turn.get("value", "").strip()

        if role == "human":
            text += f"<|im_start|>user\n{content}<|im_end|>\n"
        elif role == "gpt":
            text += f"<|im_start|>assistant\n{content}<|im_end|>\n"

    return {"text": text}


print("Loading dataset...")
dataset = load_dataset("teknium/OpenHermes-2.5", split="train")
dataset = dataset.map(format_conversation, remove_columns=dataset.column_names)
# Drop empty rows
dataset = dataset.filter(lambda x: len(x["text"]) > 50)

print(f"✅ Dataset ready: {len(dataset):,} examples")
print("\nSample:\n", dataset[0]["text"][:400])

## 🏋️ Step 8 — Configure & Run Training

Training with `SFTTrainer` from the `trl` library. At `max_steps=1000` this takes ~4–6 hours on a free T4.

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="zackai-outputs",
        save_strategy="steps",
        save_steps=250,
    ),
)

print("🚀 Starting training...")
trainer_stats = trainer.train()
print("✅ Training complete!")
print(trainer_stats)

## 💾 Step 9 — Merge LoRA & Save Model Locally

Merge the LoRA adapters back into the base model weights before export.

In [ ]:
print("Merging LoRA adapters...")
model.save_pretrained_merged("zackai-merged", tokenizer, save_method="merged_16bit")
print("✅ Merged model saved to zackai-merged/")

## 📤 Step 10 — Export to GGUF (Q4_K_M, ~4 GB)

This creates the GGUF file you can load directly into LM Studio.

In [ ]:
print("Exporting to GGUF (Q4_K_M)...")
model.save_pretrained_gguf("zackai-gguf", tokenizer, quantization_method="q4_k_m")
print("✅ GGUF file saved to zackai-gguf/")

import os
gguf_files = [f for f in os.listdir("zackai-gguf") if f.endswith(".gguf")]
for f in gguf_files:
    size_gb = os.path.getsize(os.path.join("zackai-gguf", f)) / 1e9
    print(f"  {f}  ({size_gb:.2f} GB)")

## ☁️ Step 11 — Upload to Hugging Face Hub

The full model AND the GGUF file are uploaded so you can always retrieve them.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Create the repo if it doesn't exist yet
api.create_repo(repo_id=HF_REPO, exist_ok=True)
print(f"✅ HF repo ready: https://huggingface.co/{HF_REPO}")

# Upload merged model weights
print("Uploading merged model...")
api.upload_folder(
    folder_path="zackai-merged",
    repo_id=HF_REPO,
    repo_type="model",
)
print("✅ Merged model uploaded")

# Upload GGUF file(s)
print("Uploading GGUF file...")
for gguf_file in gguf_files:
    api.upload_file(
        path_or_fileobj=os.path.join("zackai-gguf", gguf_file),
        path_in_repo=gguf_file,
        repo_id=HF_REPO,
        repo_type="model",
    )
    print(f"  ✅ {gguf_file} uploaded")

print(f"\n🎉 Done! Your model is at: https://huggingface.co/{HF_REPO}")

## 🧪 Step 12 — Test Inference

Quick sanity check to make sure ZackAI can answer a question.

In [ ]:
# Switch model to inference mode (faster)
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system",    "content": SYSTEM_PROMPT},
    {"role": "user",      "content": "What is the meaning of life?"},
]

# Build prompt using tokenizer's chat template
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("User: What is the meaning of life?")
print("ZackAI:", response)